# BONUS · B4 · AI/BI Dashboards & Genie

> *This module is optional. You can work through it independently after the main course sessions.*

The Gold layer data is ready. Now it needs to be presented to the business. Databricks offers two tools:
- **AI/BI Dashboards** — interactive dashboards built on SQL, with no need to export to external BI tools
- **AI/BI Genie** — "conversation" with data in natural language — business users ask questions in plain English and receive charts

## Topics

| # | Topic |
|---|-------|
| 1 | SQL Editor — direct queries |
| 2 | AI/BI Dashboards — creating a dashboard |
| 3 | AI/BI Genie — natural language analytics |
| 4 | End-to-end demo: Gold table → Dashboard → Genie |

## Learning Objectives

After completing this module you will be able to:

- **Use** the SQL Editor in Databricks for ad-hoc queries
- **Create** a simple AI/BI Dashboard with charts and filters
- **Publish** a dashboard for other users
- **Configure** a Genie Space and ask a question in natural language
- **Know** when to use a Dashboard and when to use Genie


## Business Context

RetailHub's Gold layer contains sales, customer, and product data built and maintained by the data engineering team. Business stakeholders — sales managers, regional directors, and the CMO — need to explore this data without writing SQL or setting up external BI tools.

**Two Databricks-native tools cover their needs:**

| Stakeholder | Need | Tool |
|-------------|------|------|
| BI analyst / report author | Build a recurring dashboard with KPIs and charts | AI/BI Dashboard |
| Business user / manager | Ask a one-off question in plain English | AI/BI Genie |

In this module we create two compact summary tables from the RetailHub Gold data and use them to demonstrate both tools end-to-end.

## Setup

In [ ]:
from pyspark.sql import functions as F

display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE",  BRONZE),
    ("SILVER",  SILVER),
    ("GOLD",    GOLD),
], ["Variable", "Value"]))

In [ ]:
from pyspark.sql import functions as F

print(f"Catalog: {CATALOG}")
print(f"Silver:  {SILVER}")
print(f"Gold:    {GOLD}")

## Preparing AI/BI Serving Tables

We create two compact Gold summary tables optimised for dashboards and Genie.
Both are built from the course Gold tables (`gold.fact_sales`, `gold.dim_customer`) and are safe to recreate at any time.

| Table | Source | Purpose |
|---|---|---|
| `gold.aibi_sales_summary` | `gold.fact_sales` + `gold.dim_customer` | Monthly revenue, orders and customers by region and category |
| `gold.aibi_top_customers` | `gold.fact_sales` + `gold.dim_customer` | Top 50 customers by total spend |

In [ ]:
required = [f"{GOLD}.fact_sales", f"{GOLD}.dim_customer"]
missing = [t for t in required if not spark.catalog.tableExists(t)]
if missing:
    for t in missing:
        print(f"[MISSING] {t}")
    raise Exception("Pre-check failed. Run data/generate_training_dataset.ipynb first.")
print("[OK] Gold source tables are available.")
display(spark.sql(f"SHOW TABLES IN {GOLD}"))

In [ ]:
aibi_sales_table = f"{GOLD}.aibi_sales_summary"

spark.sql(f"""
    CREATE OR REPLACE TABLE {aibi_sales_table}
    COMMENT 'AI/BI serving table — monthly sales by region and category. Source: gold.fact_sales + gold.dim_customer.'
    AS
    SELECT
        DATE_FORMAT(f.order_date, 'yyyy-MM') AS order_month,
        d.region,
        f.category                            AS product_category,
        COUNT(DISTINCT f.order_id)            AS total_orders,
        ROUND(SUM(f.line_revenue), 2)         AS total_revenue,
        ROUND(AVG(f.line_revenue), 2)         AS avg_order_value,
        COUNT(DISTINCT f.customer_id)         AS unique_customers
    FROM {GOLD}.fact_sales f
    JOIN {GOLD}.dim_customer d ON f.customer_id = d.customer_id
    WHERE f.status = 'Completed'
    GROUP BY DATE_FORMAT(f.order_date, 'yyyy-MM'), d.region, f.category
    ORDER BY order_month, region, product_category
""")
print(f"[OK] {aibi_sales_table}")
display(spark.table(aibi_sales_table).limit(8))

In [ ]:
aibi_customers_table = f"{GOLD}.aibi_top_customers"

spark.sql(f"""
    CREATE OR REPLACE TABLE {aibi_customers_table}
    COMMENT 'AI/BI serving table — top 50 RetailHub customers by total spend. Source: gold.fact_sales + gold.dim_customer.'
    AS
    SELECT
        d.customer_id,
        d.customer_name                      AS full_name,
        d.city,
        d.segment,
        COUNT(DISTINCT f.order_id)           AS total_orders,
        ROUND(SUM(f.line_revenue), 2)        AS total_spent
    FROM {GOLD}.fact_sales f
    JOIN {GOLD}.dim_customer d ON f.customer_id = d.customer_id
    WHERE f.status = 'Completed'
    GROUP BY d.customer_id, d.customer_name, d.city, d.segment
    ORDER BY total_spent DESC
    LIMIT 50
""")
print(f"[OK] {aibi_customers_table}")
display(spark.table(aibi_customers_table).limit(10))

## 1. SQL Editor

The SQL Editor is a **lightweight SQL interface** in Databricks — an alternative to notebooks for analysts and BI specialists who work primarily in SQL.

### How to open it:
1. Click **SQL Editor** in the left menu (or **New → SQL Editor**)
2. In the top-right corner select a **SQL Warehouse** (not a notebook cluster!)
3. Start writing SQL

![SQL Editor](../../assets/images/2faf80276f144bb6b6619ca4f8805c7f.png "2faf80276f144bb6b6619ca4f8805c7f.png")

### Try the following queries in the SQL Editor (not in this notebook):


In [ ]:
# Paste these queries into the SQL Editor in the GUI
# (running them here also works, but the goal is to get familiar with the interface)

print("Query 1 — Revenue trend by month:")
print(f"""
SELECT
    order_month,
    SUM(total_revenue) AS monthly_revenue,
    SUM(total_orders)  AS monthly_orders
FROM {aibi_sales_table}
GROUP BY order_month
ORDER BY order_month;
""")

print("Query 2 — Revenue and orders by region:")
print(f"""
SELECT
    region,
    SUM(total_revenue)        AS total_revenue,
    SUM(total_orders)         AS total_orders,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value
FROM {aibi_sales_table}
GROUP BY region
ORDER BY total_revenue DESC;
""")

In [ ]:
# Run in notebook — preview data before creating the dashboard
display(
    spark.sql(f"""
        SELECT
            order_month,
            region,
            SUM(total_revenue) AS revenue,
            SUM(total_orders)  AS orders
        FROM {aibi_sales_table}
        GROUP BY order_month, region
        ORDER BY order_month, revenue DESC
    """)
)

## 2. AI/BI Dashboards — step-by-step creation

AI/BI Dashboards replaced the old Databricks SQL Dashboards in 2024. They are faster, more polished, and support AI-assisted authoring.

### Step 1 — Create a new Dashboard
1. Left menu: **New** → **Dashboard** (or Workspace → Create → Dashboard)
2. Give it a name: `RetailHub Sales Overview — [your username]`

### Step 2 — Add the first dataset (Revenue Trend)
1. Click **Add dataset** → **Create from SQL**
2. In the SQL field enter:

```sql
SELECT order_month, SUM(total_revenue) AS revenue
FROM gold.aibi_sales_summary
GROUP BY order_month
ORDER BY order_month
```

3. Name the dataset: `Monthly Revenue`

### Step 3 — Add a visualisation based on the dataset
1. Click **Add visualization** on the `Monthly Revenue` dataset
2. Set the chart type: **Line Chart**
3. X axis: `order_month`, Y axis: `revenue`
4. Title: `Monthly Revenue Trend`

### Step 4 — Add a dataset for Revenue by Region
1. Click **Add dataset** → **Create from SQL**
2. SQL:

```sql
SELECT region, product_category, SUM(total_revenue) AS revenue
FROM gold.aibi_sales_summary
GROUP BY region, product_category
ORDER BY revenue DESC
```

3. Name the dataset: `Revenue by Region & Category`

### Step 5 — Add a bar chart visualisation
1. Click **Add visualization** on the `Revenue by Region & Category` dataset
2. Type: **Bar Chart**, Stacked
3. X: `region`, Y: `revenue`, Color: `product_category`
4. Title: `Revenue by Region & Category`

### Step 6 — Add a KPI tile
1. Click **Add dataset** → **Create from SQL**
2. SQL:

```sql
SELECT SUM(total_revenue) AS total_revenue FROM gold.aibi_sales_summary
```

3. Name the dataset: `Total Revenue`
4. Click **Add visualization** → Type: **Counter**, Title: `Total Revenue`, Format: `$#,##0`

### Step 7 — Add a filter (date picker)
1. **Add widget** → **Filter** → Drop-down
2. Title: `Month`, bind to `order_month` from your datasets

### Step 8 — Publish the dashboard
1. Click **Publish** (top-right corner)
2. Choose **Share** → add a colleague as viewer or editor
3. Tick **Schedule** → set refresh every 24 h

## 3. AI/BI Genie — Natural Language Analytics

Genie is a space ("Genie Space") where a business user can **ask questions about data in natural language** and receive ready-made charts or answers.

No SQL knowledge is required. Genie builds and executes the query automatically.

### Creating a Genie Space
1. Left menu: **New** → **Genie Space**
2. Name: `RetailHub Analytics — [your username]`
3. **Add tables** → add:
   - `{CATALOG}.gold.aibi_sales_summary`
   - `{CATALOG}.gold.aibi_top_customers`
   - `{CATALOG}.gold.dim_customer`
   - `{CATALOG}.gold.dim_product`
4. (Optional) Add **context instructions** — e.g.:
   > `total_revenue is always completed-order revenue in USD. segment values are: Regular, Premium, VIP. VIP = customers spending over 10,000 USD per year. region values: North, South, East, West, Central.`
5. Click **Save**

### Questions to test

Type each question into the Genie input field and watch how it generates a query and a chart:

In [0]:
# Sample questions you can type into Genie (these are strings, not code!)
genie_questions = [
    "Which month had the highest revenue?",
    "Show me the sales trend for Electronics in the North region",
    "How many VIP customers do we have?",
    "Which city generates the most revenue?",
    "Compare the North and South regions by number of orders",
    "What is the average spend per customer in each city?",
    "In which month did we record the largest month-over-month growth?",
]

print("Questions to test in the Genie Space:")
for i, q in enumerate(genie_questions, 1):
    print(f"  {i}. {q}")


### When Genie works best:
- Aggregation questions (`sum`, `avg`, `count`, `top N`)
- Comparisons (`North vs South`, `this month vs last month`)
- Time-based trends (`month over month`, `growth`)
- Filtering by value (`Electronics only`, `VIP customers only`)

### When Genie has limitations:
- Complex joins across many tables without good context
- Business calculations requiring additional logic (e.g., LTV, attribution)
- Queries that depend on precise knowledge of poorly named columns (e.g., `col_a_v2_final`)

> **Trainer tip:** A good column name = `total_revenue_usd`, a bad one = `c1`. Genie is only as good as the column names and context instructions.


## 4. When to use which tool — decision table

| Tool | For whom | When | Limitations |
|------|----------|------|-------------|
| **Notebook** | Data Engineers, Data Scientists | ELT, pipelines, ML, exploration | Requires Python / SQL knowledge |
| **SQL Editor** | SQL Analysts | Ad-hoc queries, debugging, preparing SQL for dashboards | No visualisation |
| **AI/BI Dashboard** | Analysts, BI, management | Regular reports, KPIs, monitoring | Static definition — questions are predictable |
| **AI/BI Genie** | Business Users, Analysts | Deep-dive, ad-hoc questions without SQL | Limited to data in the Genie Space |

> **Typical company flow:**  
> Data Engineer (Notebook) → writes to Gold Delta → Analyst (SQL Editor) → validates → AI/BI Dashboard → management views KPIs daily → Business User (Genie) → asks their own questions


In [ ]:
# display() in a notebook — how it works and how to create a chart
# ==============================================================
# After running this cell:
# 1. The result appears as a TABLE
# 2. Click the "+" icon next to "Table" (top-right of the result widget)
# 3. Choose chart type: Bar Chart
# 4. X axis: product_category, Y axis: revenue
# 5. Click "Save" — the chart will remain in the notebook

display(
    spark.sql(f"""
        SELECT
            product_category,
            SUM(total_revenue) AS revenue,
            SUM(total_orders)  AS orders
        FROM {aibi_sales_table}
        GROUP BY product_category
        ORDER BY revenue DESC
    """)
)

In [ ]:
# Revenue trend by month and region
# Click "+" → Line Chart → X: order_month, Y: revenue, Color: region

display(
    spark.sql(f"""
        SELECT
            order_month,
            region,
            SUM(total_revenue) AS revenue
        FROM {aibi_sales_table}
        GROUP BY order_month, region
        ORDER BY order_month
    """)
)

In [ ]:
for t in [f"{GOLD}.aibi_sales_summary", f"{GOLD}.aibi_top_customers"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
    print(f"[dropped] {t}")

## Cleanup

Run the cell below to remove the AI/BI serving tables created in this module.
They will be recreated the next time you run this notebook (or `generate_training_dataset.ipynb`).

## Summary

### What you accomplished:
- Created `gold.aibi_sales_summary` and `gold.aibi_top_customers` from real Gold tables
- SQL Editor → fast queries directly on governed Gold data
- AI/BI Dashboard → regular report with KPIs, charts, and a month filter
- AI/BI Genie → business user asks in plain English and gets a chart

### Key facts:
1. AI/BI Dashboards use a **SQL Warehouse** — separate, dedicated compute for BI
2. Dashboards can be **refreshed automatically** (scheduled) and **emailed**
3. Genie works best when tables have **descriptive column names** and context instructions
4. Everything is **governed by Unity Catalog** — the same permissions as in notebooks

> **RetailHub Gold tables available for Genie:**
> `gold.aibi_sales_summary` · `gold.aibi_top_customers` · `gold.dim_customer` · `gold.dim_product` · `gold.fact_sales` · `gold.kpi_daily`